In [99]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd

In [100]:
trainSet = pd.read_csv("C:/Users/Miles/Documents/ml_project_cv/Data/trainSet.csv", index_col=0, parse_dates=True)
validationSet = pd.read_csv("C:/Users/Miles/Documents/ml_project_cv/Data/validationSet.csv", index_col=0, parse_dates=True)
testSet = pd.read_csv("C:/Users/Miles/Documents/ml_project_cv/Data/testSet.csv", index_col=0, parse_dates=True)
label = "Tommorrow Up"

trainSet

,5-Day Returns,20-Day Returns,20-Day Volatility,Volume Ratio,Tommorrow Up
Date,,,,,
2016-01-04,-0.022657,-0.016505,0.011617,1.602714,True
2016-01-05,-0.018761,-0.033688,0.010621,0.823323,False
2016-01-06,-0.041369,-0.040061,0.010862,1.109206,False
2016-01-07,-0.057690,-0.056753,0.011896,1.496331,False
2016-01-08,-0.058615,-0.059792,0.011987,1.446911,True
...,...,...,...,...,...
2022-12-23,-0.000939,-0.043898,0.013353,0.693118,False
2022-12-27,0.003631,-0.032226,0.012964,0.603622,False
2022-12-28,-0.010196,-0.042609,0.013191,0.820001,True


In [ ]:
X = trainSet[
    ["5-Day Returns",
     "20-Day Returns",
     "20-Day Volatility",
     "Volume Ratio"]
]

y = trainSet[["Tommorrow Up"]]


Xtensor = torch.tensor(
    X.values,
    dtype=torch.float32
)

yTensor = torch.tensor(
    y.astype(float).values,
    dtype=torch.float32
)

dataset = TensorDataset(Xtensor, yTensor)

trainingDataset = DataLoader(dataset,batch_size=128,shuffle=True)

In [ ]:
X = validationSet[
    ["5-Day Returns",
     "20-Day Returns",
     "20-Day Volatility",
     "Volume Ratio"]
]

y = validationSet[["Tommorrow Up"]]


Xtensor = torch.tensor(
    X.values,
    dtype=torch.float32
)

yTensor = torch.tensor(
    y.astype(float).values,
    dtype=torch.float32
)

dataset = TensorDataset(Xtensor, yTensor)

validationDataset = DataLoader(dataset,batch_size=128,shuffle=True)

In [ ]:
X = testSet[
    ["5-Day Returns",
     "20-Day Returns",
     "20-Day Volatility",
     "Volume Ratio"]
]

y = testSet[["Tommorrow Up"]]


Xtensor = torch.tensor(
    X.values,
    dtype=torch.float32
)

yTensor = torch.tensor(
    y.astype(float).values,
    dtype=torch.float32
)

dataset = TensorDataset(Xtensor, yTensor)

testDataset = DataLoader(dataset,batch_size=128,shuffle=False)


In [104]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [105]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(4,32),
            nn.ReLU(),
            nn.Linear(32,8),
            nn.ReLU(),
            nn.Linear(8,1)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits
    
model = NeuralNetwork().to(device)

In [ ]:
lossFunction = nn.BCEWithLogitsLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def train(dataloader, model, lossFunction, optimiser):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = lossFunction(pred, y)

        # Backpropagation
        loss.backward()
        optimiser.step()
        optimiser.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [ ]:
def test(dataloader, model, lossFunction):
    size = len(dataloader.dataset)
    numBatches = len(dataloader)
    model.eval()
    testLoss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            testLoss += lossFunction(pred, y).item()
            correct += ((torch.sigmoid(pred) > 0.5) == y).type(torch.float).sum().item()

    testLoss /= numBatches
    correct /= size

    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {testLoss:>8f} \n")



In [ ]:
def finalTest(dataloader, model, lossFunction):
    size = len(dataloader.dataset)
    numBatches = len(dataloader)
    model.eval()
    testLoss, correct = 0, 0
    predictions = []
    
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            testLoss += lossFunction(pred, y).item()
            correct += ((torch.sigmoid(pred) > 0.5) == y).type(torch.float).sum().item()

            batchPreds = (torch.sigmoid(pred).cpu() > 0.5).int().squeeze().tolist()
            predictions.extend(batchPreds)
            
    testLoss /= numBatches
    correct /= size

    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {testLoss:>8f} \n")
    testSet["Prediction"] = predictions

In [ ]:
epochs = 1000

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(trainingDataset, model, lossFunction, optimiser)
    test(validationDataset, model, lossFunction)
print("Done!")

print("Final test performance:")
finalTest(testDataset, model, lossFunction)

testSet = testSet[["Prediction"]]

testSet.to_csv("C:/Users/Miles/Documents/ml_project_cv/Predictions/NeuralNetwork.csv")

Epoch 1
-------------------------------
loss: 0.667945  [  128/ 1762]
Test Error: 
 Accuracy: 54.0%, Avg loss: 0.687181 

Epoch 2
-------------------------------
loss: 0.682855  [  128/ 1762]
Test Error: 
 Accuracy: 54.0%, Avg loss: 0.686942 

Epoch 3
-------------------------------
loss: 0.701724  [  128/ 1762]
Test Error: 
 Accuracy: 54.8%, Avg loss: 0.686852 

Epoch 4
-------------------------------
loss: 0.677388  [  128/ 1762]
Test Error: 
 Accuracy: 56.0%, Avg loss: 0.686232 

Epoch 5
-------------------------------
loss: 0.646325  [  128/ 1762]
Test Error: 
 Accuracy: 52.4%, Avg loss: 0.686292 

Epoch 6
-------------------------------
loss: 0.663289  [  128/ 1762]
Test Error: 
 Accuracy: 53.6%, Avg loss: 0.686871 

Epoch 7
-------------------------------
loss: 0.679220  [  128/ 1762]
Test Error: 
 Accuracy: 52.4%, Avg loss: 0.687604 

Epoch 8
-------------------------------
loss: 0.692781  [  128/ 1762]
Test Error: 
 Accuracy: 55.6%, Avg loss: 0.686387 

Epoch 9
----------------